# Testauswertung

In [ ]:
#　Export: python3 -m jupyter nbconvert --to html --no-input DeineDatei.ipynb
# With this command you get an HTML export, with the code hidden

import json
import pandas as pd
import plotly.io as pio
import plotly.graph_objects as go
import plotly.express as px

pio.renderers.default='notebook' #-> make the JavaScript Code Offline available

In [ ]:
# 1. Benötige Dateien öffnen

# 1. Daten laden
df = pd.read_csv("test.csv", sep=";")

# 1.2 JSON-Datei einlesen
with open("colors.json", "r", encoding="utf-8") as f:
    farb_config = json.load(f)

# Standard-Fallbacks, falls ein Label mal nicht im JSON existiert
DEFAULT_NODE = "rgba(128, 128, 128, 0.8)"
DEFAULT_LINK = "rgba(128, 128, 128, 0.3)"

## Horizontales Balkendiagram

In [ ]:
color_dict = {
    'Netflix': '#db0000',       # Das typische Netflix-Rot
    'プライム': '#146eb4',       # Das Amazon-Prime-Blau
    'なし': "#5A5A5A",           # Schwarz für "Keinen Anbieter"
    'U-NEXT': '#000000'         # Ein weiteres Blau (oder deine Wunschfarbe)
}

fig_histo = px.bar(df,
                   x="プロバイダー",
                   labels={'count':'人数'},
                   title="ストリーミングプロバイダー",
                   color="プロバイダー",
                   color_discrete_map=color_dict)
fig_histo.update_layout(bargap=0.2)
fig_histo.show()

## Histogram des Alters

In [ ]:
fig_histo = px.histogram(df, x="才")
fig_histo.update_layout(bargap=0.2)
fig_histo.show()

## Sankey Diagram

In [ ]:
# 2. Daten vorbereiten (Zahlenwerte in lesbaren Text umwandeln)
df['Altersgruppe'] = df['才'].apply(lambda x: '18-25才' if x <= 25 else '26才以上')
df['Streaming'] = df['ストリーミング'].map({1: 'ストリーミング利用', 0: 'ストリーミング未利用'})
df['Kinopreis'] = df['映画館が高い'].map({1: '映画館が高い [はい]', 0: '映画館が高い [いいえ]'})
df['Kinobesuch'] = df['よく映画館'].map({1: 'よく映画館 [はい]', 0: 'よく映画館 [いいえ]'})
df['DVD'] = df['DVD'].map({1: 'DVD [はい]', 0: 'DVD [いいえ]'})

# Die Schritte für die Brücken
schritte = [
    ('Altersgruppe', 'どこで見る'),
    ('どこで見る', 'Streaming'),
    ('Streaming', 'Kinobesuch'),
    ('Kinobesuch', 'Kinopreis'),
    ('Kinopreis', 'DVD')
]

# 3. Einzigartige Labels für die Knoten (Balken) sammeln
alle_labels = pd.unique(df[['Altersgruppe', 'どこで見る', 'Streaming', 'Kinobesuch', 'DVD', 'Kinopreis']].values.ravel('K'))
alle_labels = [x for x in alle_labels if pd.notna(x)]
label_zu_index = {label: i for i, label in enumerate(alle_labels)}

# 4. KNOTEN-FARBEN vollautomatisch aus der JSON-Datei zuweisen
node_colors = [farb_config.get(label, {}).get("node", DEFAULT_NODE) for label in alle_labels]

# 5. Ströme UND BAND-FARBEN berechnen
sources = []
targets = []
values = []
link_labels = []
link_colors = [] 

for start_col, ziel_col in schritte:
    paar_counts = df.groupby([start_col, ziel_col]).size().reset_index(name='count')
    
    for _, row in paar_counts.iterrows():
        sources.append(label_zu_index[row[start_col]])
        targets.append(label_zu_index[row[ziel_col]])
        values.append(row['count'])
        link_labels.append(f"{row[start_col]} ➔ {row[ziel_col]}")
        
        # HIER WIRD DIE FARBE DIREKT AN DEN TEXT GEKOPPELT:
        aktueller_text = row[start_col]
        band_farbe = farb_config.get(aktueller_text, {}).get("link", DEFAULT_LINK)
        link_colors.append(band_farbe)

# 6. Das perfekt designte Sankey-Diagramm zeichnen
fig = go.Figure(data=[go.Sankey(
    valueformat = ".0f",
    valuesuffix = " 人",
    node = dict(
      pad = 15, thickness = 15,
      line = dict(color = "black", width = 0.5),
      label = alle_labels,
      color = node_colors       # Geladen aus JSON
    ),
    link = dict(
      source = sources, target = targets, value = values,
      label = link_labels,
      color = link_colors       # Geladen aus JSON
    )
)])

fig.update_layout(
    title_text="映画やシリーズを見る (N=10)",
    font_size=11, font_family="Hiragino Sans",
    width=1000, height=600
)

fig.show()

## Sunburst Diagram

In [ ]:
# 2. Altersgruppen vorbereiten (für sauberere Ringe)
df['Altersgruppe'] = df['才'].apply(lambda x: '18-25才' if x <= 25 else '26才以上')

# Map für die Farben der inneren Altersgruppen-Ringe aus deiner JSON bauen
color_discrete_map = {
    '18-25才': farb_config.get('18-25才', {}).get('node', 'rgba(31, 119, 180, 0.8)'),
    '26才以上': farb_config.get('26才以上', {}).get('node', 'rgba(255, 127, 14, 0.8)')
}

# 3. Das Sunburst-Diagramm erstellen
# Pfad geändert: Altersgruppe -> Genre -> Filmtitel
fig = px.sunburst(
    df, 
    path=["Altersgruppe", "一番ジャンル", "一番映画のタイトル"], 
    values="才",
    color="Altersgruppe",                  # Färbt das Diagramm basierend auf der Altersgruppe
    color_discrete_map=color_discrete_map  # Nutzt deine exakten JSON-Farben
)

# 4. Das Design verfeinern (Schriftarten und Textdarstellung)
fig.update_layout(
    title_text="Lieblingsgenres und -filme nach Altersgruppe",
    font_family="Hiragino Sans",          # Mac-Japanisch-Support
    title_font_size=16,
    width=700,
    height=700
)

# Text-Modus anpassen: Erzwingt, dass die Filmtitel sauber im Ring rotieren
fig.update_traces(
    textinfo="label+value",                # Zeigt den Namen UND den DVD-Wert an
    insidetextorientation="radial"         # Dreht den Text strahlenförmig (perfekt für lange Filmtitel!)
)

fig.show()

*アルトマン・ドミニク – Japan Speaking Project*